# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** "line item" or an individual item within an order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(
    orders_json, record_path='lines', meta=['order', 'vendor_id']
)

assert len(lines) == 4, f'Expected 4 rows, got {len(lines)}'
assert lines['qty'].sum() == 7, f"Expected total quantity 7, got {lines['qty'].sum()}"

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.json_normalize(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)

unmatched = joined[joined['_merge'] == 'left_only']

if len(unmatched) > 0:
  for vid, group in unmatched.groupby('vendor_id'):
    print(
        f'Unmatched vendor: {vid} with {group["qty"].sum()} total units'
        ' attached.'
    )
else:
  print(
      'In the provided payload, all vendor IDs (V-01, V-10) match vendors_json.'
  )
  print('Merge breakdown:\n', joined['_merge'].value_counts())

joined

In the provided payload, all vendor IDs (V-01, V-10) match vendors_json.
Merge breakdown:
 _merge
both          4
left_only     0
right_only    0
Name: count, dtype: int64


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
vendor_totals = (
    joined.groupby('name', dropna=False)['qty'].sum().reset_index()
)

vendor_totals

,name,qty
0,Cav Merch,1
1,Hoos Burgers,6


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
zone_totals = joined.groupby('zone', dropna=False)['qty'].sum()
print(zone_totals)

top_zone = zone_totals.dropna().idxmax()
print(
    f'\nZone selling the most items: Zone {top_zone} with {zone_totals[top_zone]} items.'
)

zone
A    7
Name: qty, dtype: int64

Zone selling the most items: Zone A with 7 items.


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

cleaned_ragged = [
    {**order, 'lines': order.get('lines', [])} for order in ragged_json
]

ragged_df = pd.json_normalize(
    cleaned_ragged, record_path='lines', meta=['order', 'vendor_id']
)

ragged_df

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

- Having one row per line item makes the dataset First Normal Form (1NF) by removing nested list structures.
- Questions aggregations for items, such as "Which item sold the highest total volume?" or "What is the average quantity sold per item?" become easy with groupby() operations.
- With one row per order, answering item-level questions requires iterating through nested JSON arrays or running slow list-comprehension parsing across every row.

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

- For, practical difference, a quantity of 0 denotes an real measurement (e.g. a free item with zero charge). A missing quantity (NaN/null) represents unknown or corrupted data.
Downstream Consequences:
- Aggregate functions like mean() include 0 in the denominator, artificially pulling down the average. NaN is correctly removed from summary statistics.
- For business logic, systems interpreting 0 might skip order fulfillment or bill $0, where missing values require flagging for review or contacting the vendor.